In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# load files
metric_df = pd.read_csv('metric.csv', low_memory=False)
trace_df = pd.read_csv('trace.csv', low_memory=False)
log_df = pd.read_csv('log.csv', low_memory=False)
error_df = pd.read_csv('error_logs.csv', low_memory=False)

# helper to convert timestamp column (seconds) to timezone-aware datetime (UTC->Asia/Shanghai)
def to_shanghai(dt_series):
    return pd.to_datetime(dt_series, unit='s', utc=True).dt.tz_convert(tz)

# convert timestamps and attach new column timestamp_dt
metric_df['timestamp_dt'] = to_shanghai(metric_df['timestamp'])
trace_df['timestamp_dt'] = to_shanghai(trace_df['timestamp'])
log_df['timestamp_dt'] = to_shanghai(log_df['timestamp'])
error_df['timestamp_dt'] = to_shanghai(error_df['timestamp'])

# Summaries for metric.csv
metric_summary = pd.DataFrame({
    'total_rows': [int(metric_df.shape[0])],
    'timestamp_min': [metric_df['timestamp_dt'].min()],
    'timestamp_max': [metric_df['timestamp_dt'].max()]
})

metric_cmdb_counts = metric_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
metric_kpi_counts = metric_df['kpi_name'].value_counts().head(20).rename_axis('kpi_name').reset_index(name='count')

# Summaries for trace.csv
trace_summary = pd.DataFrame({
    'total_rows': [int(trace_df.shape[0])],
    'timestamp_min': [trace_df['timestamp_dt'].min()],
    'timestamp_max': [trace_df['timestamp_dt'].max()]
})

trace_cmdb_counts = trace_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
trace_name_counts = trace_df['trace_name'].value_counts().head(20).rename_axis('trace_name').reset_index(name='count')

# Summaries for log.csv
log_summary = pd.DataFrame({
    'total_rows': [int(log_df.shape[0])],
    'timestamp_min': [log_df['timestamp_dt'].min()],
    'timestamp_max': [log_df['timestamp_dt'].max()]
})

log_cmdb_counts = log_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
log_name_counts = log_df['log_name'].value_counts().head(20).rename_axis('log_name').reset_index(name='count')

# Summaries for error_logs.csv (compact + first 10 rows with trimmed message)
error_summary = pd.DataFrame({
    'total_rows': [int(error_df.shape[0])],
    'timestamp_min': [error_df['timestamp_dt'].min()],
    'timestamp_max': [error_df['timestamp_dt'].max()]
})

error_cmdb_counts = error_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')

# Trim message to 120 chars and select first 10 rows (compact)
error_first10 = error_df.assign(
    message_trim=error_df['message'].astype(str).str.slice(0, 120)
)[['timestamp_dt', 'cmdb_id', 'message_trim']].head(10)

# Final compact outputs (variables shown, not full raw tables)
metric_summary, metric_cmdb_counts, metric_kpi_counts, \
trace_summary, trace_cmdb_counts, trace_name_counts, \
log_summary, log_cmdb_counts, log_name_counts, \
error_summary, error_cmdb_counts, error_first10

```
Out[1]:
```
```python
# Compact plain-English summary based on the previously computed inventory variables
# (reuses metric_summary, metric_cmdb_counts, metric_kpi_counts, trace_summary,
#  trace_cmdb_counts, trace_name_counts, log_summary, log_cmdb_counts,
#  log_name_counts, error_summary, error_cmdb_counts, error_first10)

def fmt_top(df, name_col, count_col='count', topn=10):
    if df is None or df.shape[0] == 0:
        return "None"
    rows = df.head(topn).itertuples(index=False)
    return ", ".join(f"{getattr(r, name_col)}:{getattr(r, count_col)}" for r in rows)

m_rows = int(metric_summary['total_rows'].iloc[0])
m_min = str(metric_summary['timestamp_min'].iloc[0])
m_max = str(metric_summary['timestamp_max'].iloc[0])
m_cmdb = fmt_top(metric_cmdb_counts, 'cmdb_id', 'count', topn=10)
m_kpis = fmt_top(metric_kpi_counts, 'kpi_name', 'count', topn=10)

t_rows = int(trace_summary['total_rows'].iloc[0])
t_min = str(trace_summary['timestamp_min'].iloc[0])
t_max = str(trace_summary['timestamp_max'].iloc[0])
t_cmdb = fmt_top(trace_cmdb_counts, 'cmdb_id', 'count', topn=10)
t_names = fmt_top(trace_name_counts, 'trace_name', 'count', topn=10)

l_rows = int(log_summary['total_rows'].iloc[0])
l_min = str(log_summary['timestamp_min'].iloc[0])
l_max = str(log_summary['timestamp_max'].iloc[0])
l_cmdb = fmt_top(log_cmdb_counts, 'cmdb_id', 'count', topn=10)
l_names = fmt_top(log_name_counts, 'log_name', 'count', topn=10)

e_rows = int(error_summary['total_rows'].iloc[0])
e_min = error_summary['timestamp_min'].iloc[0]
e_max = error_summary['timestamp_max'].iloc[0]
e_cmdb = fmt_top(error_cmdb_counts, 'cmdb_id', 'count', topn=20)

summary = (
    "Inventory summary (timestamps in Asia/Shanghai UTC+8):\n\n"
    "metric.csv:\n"
    f"- total rows: {m_rows}\n"
    f"- timestamp range: {m_min} to {m_max}\n"
    f"- top cmdb_id by row count (top 10): {m_cmdb}\n"
    f"- top kpi_name by row count (top 10): {m_kpis}\n\n"
    "trace.csv:\n"
    f"- total rows: {t_rows}\n"
    f"- timestamp range: {t_min} to {t_max}\n"
    f"- top cmdb_id by row count (top 10): {t_cmdb}\n"
    f"- top trace_name by row count (top 10): {t_names}\n\n"
    "log.csv:\n"
    f"- total rows: {l_rows}\n"
    f"- timestamp range: {l_min} to {l_max}\n"
    f"- top cmdb_id by row count (top 10): {l_cmdb}\n"
    f"- top log_name by row count (top 10): {l_names}\n\n"
    "error_logs.csv:\n"
    f"- total rows: {e_rows}\n"
    f"- timestamp range: {('None' if pd.isna(e_min) else str(e_min))} to {('None' if pd.isna(e_max) else str(e_max))}\n"
    f"- top cmdb_id counts (top 20): {e_cmdb}\n"
    f"- first 10 rows: {'none' if error_first10.shape[0]==0 else 'see error_first10 variable (timestamp_dt, cmdb_id, message_trim)'}\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows             timestamp_min             timestamp_max
0       29628 2021-03-25 20:00:00+08:00 2021-03-25 20:30:00+08:00,           cmdb_id  count
0         Mysql01   4769
1         Mysql02   4754
2        Tomcat02   2040
3        Tomcat03   2027
4        Tomcat04   2025
5        Tomcat01   1971
6         Redis02   1907
7            MG02   1609
8            MG01   1607
9         Redis01   1531
10           IG01   1121
11           IG02   1094
12       apache01    921
13       apache02    893
14  ServiceTest11    124
15  ServiceTest10    120
16   ServiceTest5    120
17   ServiceTest3    120
18   ServiceTest9    120
19   ServiceTest6    120,                                              kpi_name  count
0      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    356
1                  OSLinux-OSLinux_ZABBIX_Host_Uptime    354
2        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    352
3       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    352
4          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    351
5   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    345
6                         OSLinux-CPU_CPU_CPUUserTime    345
7                              OSLinux-CPU_CPU_CPUWio    345
8                          OSLinux-CPU_CPU_CPUSysTime    345
9                         OSLinux-CPU_CPU_CPUidleutil    342
10                            OSLinux-CPU_CPU_CPULoad    342
11                                             app.rr    327
12                                            app.mrt    327
13                                             app.sr    327
14                                            app.cnt    327
15       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    293
16              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    283
17       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    272
18                         OSLinux-CPU_CPU_CPUCpuUtil    272
19           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem    271,    total_rows             timestamp_min             timestamp_max
0        6981 2021-03-25 20:00:00+08:00 2021-03-25 20:30:00+08:00,      cmdb_id  count
0       MG02   1206
1       MG01   1176
2   Tomcat04    465
3       IG01    462
4   dockerB1    462
5   Tomcat03    462
6   dockerB2    462
7   Tomcat01    459
8   dockerA2    459
9   Tomcat02    459
10      IG02    456
11  dockerA1    453,                        trace_name  count
0        trace.self.duration_mean    372
1         trace.self.duration_p95    372
2            trace.self.row_count    372
3     trace.to_MG02.duration_mean    247
4      trace.to_MG02.duration_p95    247
5         trace.to_MG02.row_count    247
6     trace.to_MG01.duration_mean    240
7      trace.to_MG01.duration_p95    240
8         trace.to_MG01.row_count    240
9   trace.from_MG02.duration_mean    124
10      trace.from_MG02.row_count    124
11   trace.from_MG02.duration_p95    124
12   trace.from_IG01.duration_p95    123
13      trace.from_IG01.row_count    123
14  trace.from_IG01.duration_mean    123
15  trace.from_IG02.duration_mean    121
16   trace.from_MG01.duration_p95    121
17      trace.from_MG01.row_count    121
18  trace.from_MG01.duration_mean    121
19   trace.from_IG02.duration_p95    121,    total_rows             timestamp_min             timestamp_max
0         415 2021-03-25 20:00:00+08:00 2021-03-25 20:30:00+08:00,     cmdb_id  count
0  Tomcat04     74
1  Tomcat01     73
2  Tomcat02     72
3  Tomcat03     72
4  apache01     62
5  apache02     62,            log_name  count
0   log.error_count    186
1     log.row_count    186
2  log.gc_pause_max     43,    total_rows timestamp_min timestamp_max
0           0           NaT           NaT, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp_dt, cmdb_id, message_trim]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse metric_df and tz if present, otherwise load and prepare
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv', low_memory=False)

tz = pytz.timezone('Asia/Shanghai')

# ensure timestamp_dt column exists and is timezone-aware (Asia/Shanghai)
if 'timestamp_dt' not in metric_df.columns:
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz correctness
    if metric_df['timestamp_dt'].dt.tz is None:
        metric_df['timestamp_dt'] = metric_df['timestamp_dt'].dt.tz_localize(tz)
    else:
        metric_df['timestamp_dt'] = metric_df['timestamp_dt'].dt.tz_convert(tz)

# 1) Global aggregation (use entire series) grouped by cmdb_id and kpi_name
group_stats = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(total_points='count', global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter to incident window (inclusive) in Asia/Shanghai timezone
start = pd.Timestamp('2021-03-25 20:00:00', tz=tz)
end = pd.Timestamp('2021-03-25 20:30:00', tz=tz)
metric_window = metric_df[(metric_df['timestamp_dt'] >= start) & (metric_df['timestamp_dt'] <= end)].copy()

# 3) For each group compute points_in_window, exceed_count_window, first_exceed_timestamp
# merge global_p95 into the window rows so we can test exceedances
mw = metric_window.merge(group_stats, on=['cmdb_id', 'kpi_name'], how='left')

# mark exceedances where value > global_p95 (strict as requested)
mw['exceed_flag'] = mw['value'] > mw['global_p95']

# points in window
points = mw.groupby(['cmdb_id', 'kpi_name'], dropna=False).size().rename('points_in_window').reset_index()

# exceed counts and first exceed timestamp (if any)
if not mw[mw['exceed_flag']].empty:
    exceeds = (
        mw[mw['exceed_flag']]
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)
        .agg(exceed_count_window=('exceed_flag','sum'),
             first_exceed_timestamp=('timestamp_dt','min'))
        .reset_index()
    )
else:
    exceeds = pd.DataFrame(columns=['cmdb_id','kpi_name','exceed_count_window','first_exceed_timestamp'])

# combine all stats
result = (
    group_stats
    .merge(points, on=['cmdb_id','kpi_name'], how='left')
    .merge(exceeds, on=['cmdb_id','kpi_name'], how='left')
)

# fill missing counts with 0 and keep timestamp as NaT when absent
result['points_in_window'] = result['points_in_window'].fillna(0).astype(int)
result['exceed_count_window'] = result['exceed_count_window'].fillna(0).astype(int)
result['first_exceed_timestamp'] = pd.to_datetime(result['first_exceed_timestamp']).dt.tz_convert(tz)

# 4) Return top 20 groups ordered by exceed_count_window desc, then first_exceed_timestamp asc
result_sorted = result.sort_values(by=['exceed_count_window','first_exceed_timestamp'], ascending=[False, True])

# select required columns and limit to 20 rows for compact output
result_top20 = result_sorted[['cmdb_id','kpi_name','total_points','global_p95','points_in_window','exceed_count_window','first_exceed_timestamp']].head(20)

result_top20

```
Out[2]:
```
```python
# Compact plain-English summary of the previously returned top-20 groups (result_top20)
# This cell reuses result_top20 from prior execution and produces a short textual summary.

try:
    result_top20
except NameError:
    # fallback: load previous result if missing (shouldn't be needed in the current kernel)
    import pandas as pd
    import pytz
    metric_df = pd.read_csv('metric.csv', low_memory=False)
    tz = pytz.timezone('Asia/Shanghai')
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    group_stats = metric_df.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].agg(total_points='count', global_p95=lambda x: x.quantile(0.95)).reset_index()
    start = pd.Timestamp('2021-03-25 20:00:00', tz=tz)
    end = pd.Timestamp('2021-03-25 20:30:00', tz=tz)
    metric_window = metric_df[(metric_df['timestamp_dt'] >= start) & (metric_df['timestamp_dt'] <= end)].copy()
    mw = metric_window.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')
    mw['exceed_flag'] = mw['value'] > mw['global_p95']
    points = mw.groupby(['cmdb_id','kpi_name'], dropna=False).size().rename('points_in_window').reset_index()
    if not mw[mw['exceed_flag']].empty:
        exceeds = mw[mw['exceed_flag']].groupby(['cmdb_id','kpi_name'], dropna=False).agg(exceed_count_window=('exceed_flag','sum'), first_exceed_timestamp=('timestamp_dt','min')).reset_index()
    else:
        exceeds = pd.DataFrame(columns=['cmdb_id','kpi_name','exceed_count_window','first_exceed_timestamp'])
    result = group_stats.merge(points, on=['cmdb_id','kpi_name'], how='left').merge(exceeds, on=['cmdb_id','kpi_name'], how='left')
    result['points_in_window'] = result['points_in_window'].fillna(0).astype(int)
    result['exceed_count_window'] = result['exceed_count_window'].fillna(0).astype(int)
    result['first_exceed_timestamp'] = pd.to_datetime(result['first_exceed_timestamp']).dt.tz_convert(tz)
    result_sorted = result.sort_values(by=['exceed_count_window','first_exceed_timestamp'], ascending=[False, True])
    result_top20 = result_sorted[['cmdb_id','kpi_name','total_points','global_p95','points_in_window','exceed_count_window','first_exceed_timestamp']].head(20)

# Build summary
count_groups = result_top20.shape[0]
groups_with_exceed = (result_top20['exceed_count_window'] > 0).sum()
earliest = result_top20['first_exceed_timestamp'].dropna().min()
latest = result_top20['first_exceed_timestamp'].dropna().max()

# Prepare a short list of the top groups (up to 10) with key fields
rows = []
for _, r in result_top20.head(10).iterrows():
    ts = r['first_exceed_timestamp']
    ts_str = str(ts) if pd.notna(ts) else "None"
    rows.append(f"{r['cmdb_id']} | {r['kpi_name']} | exceed_count={int(r['exceed_count_window'])} | first_exceed={ts_str}")

summary = (
    f"Summary of metric exceedances in 2021-03-25 20:00–20:30 (Asia/Shanghai):\n"
    f"- Returned groups: {count_groups} (top 20)\n"
    f"- Groups with exceedances in window: {groups_with_exceed}\n"
    f"- Exceedance counts among top groups are all 2 (sorted by earliest exceed time).\n"
    f"- Earliest first exceed timestamp in this set: {str(earliest)}\n"
    f"- Latest first exceed timestamp in this set: {str(latest)}\n\n"
    f"Top group examples (component | kpi | exceed_count | first_exceed):\n"
    + "\n".join(rows)
    + "\n\n"
    "Notes: Exceedances are defined as window values > the group's global 95th percentile (computed over the full series). "
    "The listed groups are mostly CPU-related KPIs (OSLinux CPU metrics) and a few JVM/Mysql metrics."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   kpi_name  total_points    global_p95  points_in_window  exceed_count_window    first_exceed_timestamp
197       MG01                 OSLinux-CPU_CPU_CPUSysTime            31  7.277500e-01                31                    2 2021-03-25 20:00:00+08:00
198       MG01                OSLinux-CPU_CPU_CPUUserTime            31  2.581026e+01                31                    2 2021-03-25 20:00:00+08:00
300       MG02                    OSLinux-CPU_CPU_CPULoad            31  1.525000e+00                31                    2 2021-03-25 20:00:00+08:00
1236  Tomcat02                    OSLinux-CPU_CPU_CPULoad            31  1.190000e+00                31                    2 2021-03-25 20:00:00+08:00
1239  Tomcat02                     OSLinux-CPU_CPU_CPUWio            31  4.427995e-01                31                    2 2021-03-25 20:00:00+08:00
1359  Tomcat03                 OSLinux-CPU_CPU_CPUSysTime            30  5.827200e-01                30                    2 2021-03-25 20:00:00+08:00
961    Redis02                     OSLinux-CPU_CPU_CPUWio            31  2.086300e-02                31                    2 2021-03-25 20:01:00+08:00
1482  Tomcat04                 OSLinux-CPU_CPU_CPUSysTime            31  7.714380e-01                31                    2 2021-03-25 20:01:00+08:00
2         IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed            31  8.659988e+08                31                    2 2021-03-25 20:02:00+08:00
6         IG01   JVM-Threads_7778_JVM_ThreadCount_Threads            31  7.550000e+01                31                    2 2021-03-25 20:02:00+08:00
303       MG02                     OSLinux-CPU_CPU_CPUWio            31  3.786500e-01                31                    2 2021-03-25 20:02:00+08:00
1485  Tomcat04                OSLinux-CPU_CPU_CPUidleutil            31  7.342106e+01                31                    2 2021-03-25 20:02:00+08:00
66        IG01      OSLinux-OSLinux_MEMORY_MEMORY_UserMem            22  7.669950e+03                22                    2 2021-03-25 20:04:00+08:00
286       MG02  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed            31  9.073892e+08                31                    2 2021-03-25 20:04:00+08:00
304       MG02                OSLinux-CPU_CPU_CPUidleutil            31  7.331075e+01                31                    2 2021-03-25 20:04:00+08:00
736    Mysql02            Mysql-MySQL_3306_ThreadsRunning            30  6.100000e+00                30                    2 2021-03-25 20:04:00+08:00
290       MG02   JVM-Threads_7779_JVM_ThreadCount_Threads            31  7.350000e+01                31                    2 2021-03-25 20:05:00+08:00
1237  Tomcat02                 OSLinux-CPU_CPU_CPUSysTime            31  7.152595e-01                31                    2 2021-03-25 20:05:00+08:00
1360  Tomcat03                OSLinux-CPU_CPU_CPUUserTime            30  2.566378e+01                30                    2 2021-03-25 20:05:00+08:00
1483  Tomcat04                OSLinux-CPU_CPU_CPUUserTime            31  2.588571e+01                31                    2 2021-03-25 20:05:00+08:00```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse 'result' if available; otherwise recompute group-level stats as in previous step
try:
    result
except NameError:
    # load metric and compute group-level stats (global p95 etc.)
    metric_df = pd.read_csv('metric.csv', low_memory=False)
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    group_stats = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
        .agg(total_points='count', global_p95=lambda x: x.quantile(0.95))
        .reset_index()
    )
    start = pd.Timestamp('2021-03-25 20:00:00', tz=tz)
    end = pd.Timestamp('2021-03-25 20:30:00', tz=tz)
    metric_window = metric_df[(metric_df['timestamp_dt'] >= start) & (metric_df['timestamp_dt'] <= end)].copy()
    mw = metric_window.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')
    mw['exceed_flag'] = mw['value'] > mw['global_p95']
    points = mw.groupby(['cmdb_id','kpi_name'], dropna=False).size().rename('points_in_window').reset_index()
    if not mw[mw['exceed_flag']].empty:
        exceeds = mw[mw['exceed_flag']].groupby(['cmdb_id','kpi_name'], dropna=False).agg(
            exceed_count_window=('exceed_flag','sum'),
            first_exceed_timestamp=('timestamp_dt','min')
        ).reset_index()
    else:
        exceeds = pd.DataFrame(columns=['cmdb_id','kpi_name','exceed_count_window','first_exceed_timestamp'])
    result = group_stats.merge(points, on=['cmdb_id','kpi_name'], how='left').merge(exceeds, on=['cmdb_id','kpi_name'], how='left')
    result['points_in_window'] = result['points_in_window'].fillna(0).astype(int)
    result['exceed_count_window'] = result['exceed_count_window'].fillna(0).astype(int)
    result['first_exceed_timestamp'] = pd.to_datetime(result['first_exceed_timestamp']).dt.tz_convert(tz)

# Now aggregate per-component
# distinct_kpis_with_exceed: count of kpi_name where exceed_count_window > 0
exceed_rows = result[result['exceed_count_window'] > 0].copy()

per_comp_exceed_counts = (
    exceed_rows
    .groupby('cmdb_id', dropna=False)
    .agg(
        distinct_kpis_with_exceed=('kpi_name', 'nunique'),
        earliest_exceed_timestamp=('first_exceed_timestamp', 'min')
    )
    .reset_index()
)

# total_exceed_points and total_points_in_window across all kpis (use result)
per_comp_sums = (
    result
    .groupby('cmdb_id', dropna=False)
    .agg(
        total_exceed_points=('exceed_count_window','sum'),
        total_points_in_window=('points_in_window','sum')
    )
    .reset_index()
)

# merge summaries
comp_summary = per_comp_sums.merge(per_comp_exceed_counts, on='cmdb_id', how='left')

# fill NaN distinct_kpis_with_exceed with 0 and keep earliest_exceed_timestamp as NaT where no exceed
comp_summary['distinct_kpis_with_exceed'] = comp_summary['distinct_kpis_with_exceed'].fillna(0).astype(int)
# ensure integer types for counts
comp_summary['total_exceed_points'] = comp_summary['total_exceed_points'].astype(int)
comp_summary['total_points_in_window'] = comp_summary['total_points_in_window'].astype(int)

# order by total_exceed_points desc, then earliest_exceed_timestamp asc (NaT will sort last)
comp_summary_sorted = comp_summary.sort_values(by=['total_exceed_points','earliest_exceed_timestamp'], ascending=[False, True])

# select required columns and limit to top 20
comp_summary_top20 = comp_summary_sorted[['cmdb_id','distinct_kpis_with_exceed','total_exceed_points','total_points_in_window','earliest_exceed_timestamp']].head(20)

comp_summary_top20

```
Out[3]:
```
```python
# Produce a concise plain-English summary string using the computed comp_summary_top20
# Reuse existing variable if present; otherwise, raise informative error.

try:
    comp_summary_top20
except NameError:
    raise RuntimeError("comp_summary_top20 not found in the kernel. Please run the aggregation step first.")

# Build summary
n = comp_summary_top20.shape[0]
rows = []
for _, r in comp_summary_top20.head(10).iterrows():
    ts = r['earliest_exceed_timestamp']
    ts_str = str(ts) if pd.notna(ts) else "None"
    rows.append(f"{r['cmdb_id']}: distinct_kpis_with_exceed={int(r['distinct_kpis_with_exceed'])}, "
                f"total_exceed_points={int(r['total_exceed_points'])}, "
                f"total_points_in_window={int(r['total_points_in_window'])}, "
                f"earliest_exceed={ts_str}")

summary = (
    f"Per-component exceedance summary for 2021-03-25 20:00–20:30 (Asia/Shanghai):\n"
    f"- Components returned: {n} (top 20)\n"
    f"- Top components by total_exceed_points:\n"
    + "\n".join(rows)
    + "\n\nNotes: Exceedances are counts of measurements in the window that exceed each group's global 95th percentile (computed over the full series). "
    "Databases (Mysql01/Mysql02), management nodes (MG01/MG02), Tomcats, IG, Redis and apache appear among the most affected components."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  distinct_kpis_with_exceed  total_exceed_points  total_points_in_window earliest_exceed_timestamp
5         Mysql02                        103                  152                    4754 2021-03-25 20:01:00+08:00
4         Mysql01                         94                  135                    4769 2021-03-25 20:00:00+08:00
3            MG02                         47                   56                    1609 2021-03-25 20:00:00+08:00
19       Tomcat01                         46                   54                    1971 2021-03-25 20:00:00+08:00
21       Tomcat03                         43                   52                    2027 2021-03-25 20:00:00+08:00
2            MG01                         42                   50                    1607 2021-03-25 20:00:00+08:00
20       Tomcat02                         42                   50                    2040 2021-03-25 20:00:00+08:00
22       Tomcat04                         40                   48                    2025 2021-03-25 20:01:00+08:00
0            IG01                         38                   42                    1121 2021-03-25 20:00:00+08:00
7         Redis02                         33                   38                    1907 2021-03-25 20:00:00+08:00
1            IG02                         34                   36                    1094 2021-03-25 20:00:00+08:00
23       apache01                         30                   30                     921 2021-03-25 20:00:00+08:00
24       apache02                         29                   29                     893 2021-03-25 20:00:00+08:00
6         Redis01                         28                   28                    1531 2021-03-25 20:04:00+08:00
25       dockerA2                          4                    4                      51 2021-03-25 20:04:00+08:00
15   ServiceTest6                          2                    4                     120 2021-03-25 20:07:00+08:00
18   ServiceTest9                          2                    4                     120 2021-03-25 20:07:00+08:00
8    ServiceTest1                          2                    4                     116 2021-03-25 20:10:00+08:00
9   ServiceTest10                          2                    4                     120 2021-03-25 20:12:00+08:00
16   ServiceTest7                          2                    4                     120 2021-03-25 20:12:00+08:00```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse trace_df if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv', low_memory=False)

# ensure timestamp_dt exists and is timezone-aware (Asia/Shanghai)
if 'timestamp_dt' not in trace_df.columns:
    trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if trace_df['timestamp_dt'].dt.tz is None:
        trace_df['timestamp_dt'] = trace_df['timestamp_dt'].dt.tz_localize(tz)
    else:
        trace_df['timestamp_dt'] = trace_df['timestamp_dt'].dt.tz_convert(tz)

# 1) Global aggregation over full trace.csv by cmdb_id and trace_name
trace_group_stats = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(total_points='count', global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter to incident window (inclusive) in Asia/Shanghai
start = pd.Timestamp('2021-03-25 20:00:00', tz=tz)
end = pd.Timestamp('2021-03-25 20:30:00', tz=tz)
trace_window = trace_df[(trace_df['timestamp_dt'] >= start) & (trace_df['timestamp_dt'] <= end)].copy()

# 3) Compute points_in_window, exceed_count_window, first_exceed_timestamp per group
tw = trace_window.merge(trace_group_stats, on=['cmdb_id', 'trace_name'], how='left')
tw['exceed_flag'] = tw['value'] > tw['global_p95']

points = tw.groupby(['cmdb_id', 'trace_name'], dropna=False).size().rename('points_in_window').reset_index()

if not tw[tw['exceed_flag']].empty:
    exceeds = (
        tw[tw['exceed_flag']]
        .groupby(['cmdb_id', 'trace_name'], dropna=False)
        .agg(exceed_count_window=('exceed_flag', 'sum'),
             first_exceed_timestamp=('timestamp_dt', 'min'))
        .reset_index()
    )
else:
    exceeds = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'exceed_count_window', 'first_exceed_timestamp'])

# merge stats
trace_result = trace_group_stats.merge(points, on=['cmdb_id', 'trace_name'], how='left').merge(exceeds, on=['cmdb_id', 'trace_name'], how='left')

# fill missing numeric values with 0 and ensure types
trace_result['points_in_window'] = trace_result['points_in_window'].fillna(0).astype(int)
trace_result['exceed_count_window'] = trace_result['exceed_count_window'].fillna(0).astype(int)
trace_result['first_exceed_timestamp'] = pd.to_datetime(trace_result['first_exceed_timestamp']).dt.tz_convert(tz)

# 4) Sort and return top 20 groups
trace_result_sorted = trace_result.sort_values(by=['exceed_count_window', 'first_exceed_timestamp'], ascending=[False, True])

trace_result_top20 = trace_result_sorted[['cmdb_id','trace_name','total_points','global_p95','points_in_window','exceed_count_window','first_exceed_timestamp']].head(20)

trace_result_top20

```
Out[4]:
```
```python
# Plain-English summary of the trace exceedance results (uses trace_result_top20 from prior computation)
try:
    trace_result_top20
except NameError:
    raise RuntimeError("trace_result_top20 not found. Please run the trace aggregation step first.")

n = trace_result_top20.shape[0]
unique_components = trace_result_top20['cmdb_id'].nunique()
exceed_values = sorted(trace_result_top20['exceed_count_window'].unique())
earliest = trace_result_top20['first_exceed_timestamp'].dropna().min()
latest = trace_result_top20['first_exceed_timestamp'].dropna().max()

examples = []
for _, r in trace_result_top20.head(10).iterrows():
    ts = r['first_exceed_timestamp']
    ts_str = str(ts) if pd.notna(ts) else "None"
    examples.append(f"{r['cmdb_id']} | {r['trace_name']} | exceeds={int(r['exceed_count_window'])} | first_exceed={ts_str}")

summary = (
    f"Trace exceedance summary for 2021-03-25 20:00–20:30 (Asia/Shanghai):\n"
    f"- Returned groups: {n} (top 20)\n"
    f"- Involved components: {unique_components}\n"
    f"- Observed exceed_count_window values (in this set): {exceed_values}\n"
    f"- Earliest first exceed among top groups: {earliest}\n"
    f"- Latest first exceed among top groups: {latest}\n\n"
    f"- Top example groups (component | trace_name | exceed_count | first_exceed):\n"
    + "\n".join(examples)
    + "\n\nNotes: The top groups are mostly inter-service trace row_count and duration p95/mean metrics (e.g., trace.from/to and duration_p95/mean), "
    "involving MG01/MG02, dockerA2/dockerB2, IG01/IG02, and Tomcat04. Exceedances are defined as window values > group global 95th percentile (computed over the full series)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name  total_points  global_p95  points_in_window  exceed_count_window    first_exceed_timestamp
53       MG01      trace.from_dockerB2.row_count            30   77.100000                30                    2 2021-03-25 20:03:00+08:00
233  dockerB2            trace.to_MG01.row_count            30   77.100000                30                    2 2021-03-25 20:03:00+08:00
14       IG01        trace.to_Tomcat04.row_count            31  123.500000                31                    2 2021-03-25 20:04:00+08:00
28       IG02     trace.to_Tomcat04.duration_p95            30    2.096595                30                    2 2021-03-25 20:04:00+08:00
40       MG01   trace.from_Tomcat04.duration_p95            31    1.466550                31                    2 2021-03-25 20:04:00+08:00
84       MG02  trace.from_dockerA2.duration_mean            31    0.107208                31                    2 2021-03-25 20:04:00+08:00
85       MG02   trace.from_dockerA2.duration_p95            31    0.769075                31                    2 2021-03-25 20:04:00+08:00
90       MG02  trace.from_dockerB2.duration_mean            31    0.081175                31                    2 2021-03-25 20:04:00+08:00
91       MG02   trace.from_dockerB2.duration_p95            31    0.657775                31                    2 2021-03-25 20:04:00+08:00
158  Tomcat04          trace.from_IG01.row_count            31  123.500000                31                    2 2021-03-25 20:04:00+08:00
160  Tomcat04       trace.from_IG02.duration_p95            30    2.096595                30                    2 2021-03-25 20:04:00+08:00
169  Tomcat04         trace.to_MG01.duration_p95            31    1.466550                31                    2 2021-03-25 20:04:00+08:00
201  dockerA2        trace.to_MG02.duration_mean            31    0.107208                31                    2 2021-03-25 20:04:00+08:00
202  dockerA2         trace.to_MG02.duration_p95            31    0.769075                31                    2 2021-03-25 20:04:00+08:00
234  dockerB2        trace.to_MG02.duration_mean            31    0.081175                31                    2 2021-03-25 20:04:00+08:00
235  dockerB2         trace.to_MG02.duration_p95            31    0.657775                31                    2 2021-03-25 20:04:00+08:00
38       MG01      trace.from_Tomcat03.row_count            30  133.550000                30                    2 2021-03-25 20:05:00+08:00
45       MG01  trace.from_dockerA2.duration_mean            30    0.078937                30                    2 2021-03-25 20:05:00+08:00
46       MG01   trace.from_dockerA2.duration_p95            30    0.618760                30                    2 2021-03-25 20:05:00+08:00
60       MG01    trace.to_dockerA2.duration_mean            30    0.107924                30                    2 2021-03-25 20:05:00+08:00```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse log_df if present, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv', low_memory=False)

# ensure timestamp_dt exists and is timezone-aware (Asia/Shanghai)
if 'timestamp_dt' not in log_df.columns:
    log_df['timestamp_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # if exists but naive, localize; otherwise convert to tz
    if log_df['timestamp_dt'].dt.tz is None:
        log_df['timestamp_dt'] = log_df['timestamp_dt'].dt.tz_localize(tz)
    else:
        log_df['timestamp_dt'] = log_df['timestamp_dt'].dt.tz_convert(tz)

# incident window (inclusive)
start = pd.Timestamp('2021-03-25 20:00:00', tz=tz)
end = pd.Timestamp('2021-03-25 20:30:00', tz=tz)

components = ['Tomcat04', 'MG01', 'MG02', 'IG01', 'IG02', 'Mysql01', 'Mysql02']

# filter to window and components
log_window = log_df[
    (log_df['timestamp_dt'] >= start) &
    (log_df['timestamp_dt'] <= end) &
    (log_df['cmdb_id'].isin(components))
].copy()

# 1) Per-component aggregates
# total_log_rows_in_window
counts = log_window.groupby('cmdb_id', dropna=False).size().rename('total_log_rows_in_window').reset_index()

# sum_error_count: sum of value where log_name == 'log.error_count' within window (per component)
errors = (
    log_window[log_window['log_name'] == 'log.error_count']
    .groupby('cmdb_id', dropna=False)['value']
    .sum()
    .rename('sum_error_count')
    .reset_index()
)

# earliest_error_timestamp: earliest timestamp_dt where log_name == 'log.error_count' and value > 0
err_pos = log_window[(log_window['log_name'] == 'log.error_count') & (log_window['value'] > 0)].copy()
if not err_pos.empty:
    earliest = err_pos.groupby('cmdb_id', dropna=False)['timestamp_dt'].min().rename('earliest_error_timestamp').reset_index()
else:
    earliest = pd.DataFrame(columns=['cmdb_id','earliest_error_timestamp'])

# combine into one table for the requested components, ensure all components present
comp_agg = pd.DataFrame({'cmdb_id': components})
comp_agg = comp_agg.merge(counts, on='cmdb_id', how='left').merge(errors, on='cmdb_id', how='left').merge(earliest, on='cmdb_id', how='left')

comp_agg['total_log_rows_in_window'] = comp_agg['total_log_rows_in_window'].fillna(0).astype(int)
comp_agg['sum_error_count'] = comp_agg['sum_error_count'].fillna(0)

# Fix earliest_error_timestamp conversion robustly (handle NaT and tz-naive)
def _fix_ts(x):
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, pd.Timestamp):
        if x.tzinfo is None:
            return x.tz_localize(tz)
        else:
            return x.tz_convert(tz)
    # fallback
    return pd.to_datetime(x).tz_localize(tz)

if 'earliest_error_timestamp' in comp_agg.columns:
    comp_agg['earliest_error_timestamp'] = comp_agg['earliest_error_timestamp'].apply(_fix_ts)

# 2) Up to first 20 raw log rows in-window for these components, ordered by timestamp asc
raw_rows = log_window.sort_values('timestamp_dt').loc[:, ['timestamp_dt','cmdb_id','log_name','value']].head(20).copy()
raw_rows = raw_rows.rename(columns={'timestamp_dt': 'timestamp'})

# Return compact results (<=20 rows each)
comp_agg, raw_rows

```
Out[5]:
```
```python
# Compact plain-English summary of the log analysis (uses comp_agg and raw_rows from prior execution)
try:
    comp_agg, raw_rows
except NameError:
    raise RuntimeError("Required variables not found. Please run the log aggregation step first.")

import pandas as pd

# Build summary
lines = []
lines.append("Log analysis for 2021-03-25 20:00:00–20:30:00 (Asia/Shanghai):")
lines.append("")
lines.append("Per-component aggregates (requested components):")
for _, r in comp_agg.iterrows():
    ts = r['earliest_error_timestamp']
    ts_str = str(ts) if pd.notna(ts) else "None"
    lines.append(f"- {r['cmdb_id']}: total_log_rows_in_window={int(r['total_log_rows_in_window'])}, "
                 f"sum_error_count={float(r['sum_error_count'])}, earliest_error_timestamp={ts_str}")

lines.append("")
# Raw rows summary
n_raw = raw_rows.shape[0]
if n_raw > 0:
    first_ts = raw_rows['timestamp'].min()
    last_ts = raw_rows['timestamp'].max()
    lines.append(f"Raw log rows returned (up to 20): {n_raw} rows, timestamps from {first_ts} to {last_ts}.")
    # note about contents
    lines.append("Sample content: entries are from Tomcat04 and show log.error_count values = 0.00, log.row_count, and log.gc_pause_max.")
else:
    lines.append("No raw log rows found in the window for the requested components.")

lines.append("")
lines.append("Key conclusion:")
lines.append("- Only Tomcat04 produced log rows in the incident window (74 rows).")
lines.append("- No log.error_count > 0 was observed for Tomcat04 or any of the listed components in this window (sum_error_count = 0 for all).")
lines.append("- Therefore, there were no positive error_count log events for Tomcat04, MG01, MG02, IG01, IG02, Mysql01, or Mysql02 during the 20:00–20:30 window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  total_log_rows_in_window  sum_error_count earliest_error_timestamp
0  Tomcat04                        74              0.0                      NaT
1      MG01                         0              0.0                      NaT
2      MG02                         0              0.0                      NaT
3      IG01                         0              0.0                      NaT
4      IG02                         0              0.0                      NaT
5   Mysql01                         0              0.0                      NaT
6   Mysql02                         0              0.0                      NaT,                     timestamp   cmdb_id          log_name   value
7   2021-03-25 20:00:00+08:00  Tomcat04   log.error_count    0.00
8   2021-03-25 20:00:00+08:00  Tomcat04     log.row_count  236.00
19  2021-03-25 20:01:00+08:00  Tomcat04   log.error_count    0.00
20  2021-03-25 20:01:00+08:00  Tomcat04  log.gc_pause_max    0.05
21  2021-03-25 20:01:00+08:00  Tomcat04     log.row_count  226.00
34  2021-03-25 20:02:00+08:00  Tomcat04   log.error_count    0.00
35  2021-03-25 20:02:00+08:00  Tomcat04     log.row_count  185.00
48  2021-03-25 20:03:00+08:00  Tomcat04     log.row_count  171.00
47  2021-03-25 20:03:00+08:00  Tomcat04   log.error_count    0.00
60  2021-03-25 20:04:00+08:00  Tomcat04   log.error_count    0.00
61  2021-03-25 20:04:00+08:00  Tomcat04  log.gc_pause_max    0.09
62  2021-03-25 20:04:00+08:00  Tomcat04     log.row_count  213.00
74  2021-03-25 20:05:00+08:00  Tomcat04   log.error_count    0.00
75  2021-03-25 20:05:00+08:00  Tomcat04     log.row_count  243.00
88  2021-03-25 20:06:00+08:00  Tomcat04  log.gc_pause_max    0.06
87  2021-03-25 20:06:00+08:00  Tomcat04   log.error_count    0.00
89  2021-03-25 20:06:00+08:00  Tomcat04     log.row_count  199.00
102 2021-03-25 20:07:00+08:00  Tomcat04     log.row_count  236.00
101 2021-03-25 20:07:00+08:00  Tomcat04   log.error_count    0.00
114 2021-03-25 20:08:00+08:00  Tomcat04   log.error_count    0.00)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')
components = ['Mysql01', 'Mysql02']

# reuse metric_df if available; otherwise load and prepare
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv', low_memory=False)

# ensure timestamp_dt exists and is timezone-aware (Asia/Shanghai)
if 'timestamp_dt' not in metric_df.columns:
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # normalize existing column to timezone-aware Asia/Shanghai
    def to_shanghai(ts):
        if pd.isna(ts):
            return pd.NaT
        if isinstance(ts, pd.Timestamp):
            if ts.tzinfo is None:
                return ts.tz_localize(tz)
            else:
                return ts.tz_convert(tz)
        # fallback
        return pd.to_datetime(ts, utc=True).tz_convert(tz)
    metric_df['timestamp_dt'] = metric_df['timestamp_dt'].apply(to_shanghai)

# Filter metric rows to Mysql01 and Mysql02 for global stats
mdf = metric_df[metric_df['cmdb_id'].isin(components)].copy()

# 1) Global statistics per (cmdb_id, kpi_name)
group_stats = (
    mdf
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(total_points='count', global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter to incident window (inclusive)
start = pd.Timestamp('2021-03-25 20:00:00', tz=tz)
end = pd.Timestamp('2021-03-25 20:30:00', tz=tz)
window_df = mdf[(mdf['timestamp_dt'] >= start) & (mdf['timestamp_dt'] <= end)].copy()

# Merge global p95 into window rows
w = window_df.merge(group_stats, on=['cmdb_id', 'kpi_name'], how='left')

# mark exceedances (strict > global_p95)
w['exceed_flag'] = w['value'] > w['global_p95']

# points_in_window per group
points = w.groupby(['cmdb_id', 'kpi_name'], dropna=False).size().rename('points_in_window').reset_index()

# exceed_count_window and first_exceed_timestamp per group (only from exceed rows)
if not w[w['exceed_flag']].empty:
    exceed_specific = (
        w[w['exceed_flag']]
        .groupby(['cmdb_id','kpi_name'], dropna=False)
        .agg(
            exceed_count_window=('exceed_flag','sum'),
            first_exceed_timestamp=('timestamp_dt','min')
        )
        .reset_index()
    )
else:
    exceed_specific = pd.DataFrame(columns=['cmdb_id','kpi_name','exceed_count_window','first_exceed_timestamp'])

# max_value_in_window per group (even if no exceed)
max_vals = w.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].max().rename('max_value_in_window').reset_index()

# combine stats
summary = group_stats.merge(points, on=['cmdb_id','kpi_name'], how='left') \
                     .merge(exceed_specific, on=['cmdb_id','kpi_name'], how='left') \
                     .merge(max_vals, on=['cmdb_id','kpi_name'], how='left')

# fill missing numeric window counts with 0
summary['points_in_window'] = summary['points_in_window'].fillna(0).astype(int)
summary['exceed_count_window'] = summary['exceed_count_window'].fillna(0).astype(int)

# ensure first_exceed_timestamp has proper tz (it should come from timestamp_dt which is tz-aware)
def ensure_tz(ts):
    if pd.isna(ts):
        return pd.NaT
    if isinstance(ts, pd.Timestamp):
        if ts.tzinfo is None:
            return ts.tz_localize(tz)
        else:
            return ts.tz_convert(tz)
    return pd.to_datetime(ts).tz_localize(tz)

summary['first_exceed_timestamp'] = summary['first_exceed_timestamp'].apply(ensure_tz)

# 3) Compute max_over_p95_ratio for groups with exceed_count_window > 0
def compute_ratio(row):
    gp95 = row['global_p95']
    maxv = row['max_value_in_window']
    if row['exceed_count_window'] <= 0:
        return np.nan
    if pd.isna(gp95) or gp95 == 0:
        return np.nan
    if pd.isna(maxv):
        return np.nan
    return (maxv - gp95) / gp95

summary['max_over_p95_ratio'] = summary.apply(compute_ratio, axis=1)

# 4) Filter to rows with exceed_count_window > 0 and components Mysql01/Mysql02, order and limit
out = summary[(summary['exceed_count_window'] > 0) & (summary['cmdb_id'].isin(components))].copy()

out_sorted = out.sort_values(by=['max_over_p95_ratio','exceed_count_window'], ascending=[False, False])

result_cols = ['cmdb_id','kpi_name','total_points','global_p95','points_in_window','exceed_count_window','first_exceed_timestamp','max_value_in_window','max_over_p95_ratio']
mysql_exceed_summary = out_sorted.loc[:, result_cols].head(20)

mysql_exceed_summary

```
Out[6]:
```
```python
# Plain-English summary of MySQL metric exceedances (uses mysql_exceed_summary from prior computation)
try:
    mysql_exceed_summary
except NameError:
    raise RuntimeError("mysql_exceed_summary not found. Please run the previous aggregation step.")

# Build concise summary
n_rows = mysql_exceed_summary.shape[0]
earliest = mysql_exceed_summary['first_exceed_timestamp'].min()
latest = mysql_exceed_summary['first_exceed_timestamp'].max()

examples = []
for _, r in mysql_exceed_summary.head(6).iterrows():
    ts = r['first_exceed_timestamp']
    ts_str = str(ts) if pd.notna(ts) else "None"
    examples.append(
        f"{r['cmdb_id']} | {r['kpi_name']} | total_points={int(r['total_points'])} | "
        f"global_p95={r['global_p95']:.6g} | points_in_window={int(r['points_in_window'])} | "
        f"exceed_count={int(r['exceed_count_window'])} | first_exceed={ts_str} | "
        f"max_value={r['max_value_in_window']:.6g} | ratio={r['max_over_p95_ratio']:.6g}"
    )

summary = (
    f"MySQL metric exceedance summary (2021-03-25 20:00–20:30 Asia/Shanghai):\n"
    f"- Rows with exceedances: {n_rows}\n"
    f"- Earliest first exceed among these groups: {earliest}\n"
    f"- Latest first exceed among these groups:  {latest}\n\n"
    f"- Top examples (component | kpi | total_points | global_p95 | points_in_window | exceed_count | first_exceed | max_value | max_over_p95_ratio):\n"
    + "\n".join(examples)
    + "\n\nKey conclusions:\n"
    "- Mysql01 exhibits the largest anomaly: 'Innodb buffer pool pages dirty' spiked from a 95th percentile of 14 to a max of 85 (ratio ~5.07) with an exceed at 2021-03-25 20:30:00+08:00.\n"
    "- Mysql01 also shows lock-related metrics (Max trx rows locked, MaxTrxRowsModified, max trx lock memory bytes) exceeding their p95 around 20:07 with ratio ~0.82.\n"
    "- Both Mysql01 and Mysql02 have several disk I/O and related metrics exceeding p95 (single-point exceeds) with moderate ratios (≈0.4–0.82).\n"
    "- Mysql02 shows ThreadsRunning exceeding twice (max 10 vs p95 6.1, ratio ~0.64) with first exceed at 20:04.\n\n"
    "Interpretation: the most severe signal is on Mysql01 (dirty buffer pages), accompanied by increased locks and some disk I/O spikes on both MySQL nodes. These point to possible MySQL workload/backlog and I/O pressure during the incident window."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points  global_p95  points_in_window  exceed_count_window    first_exceed_timestamp  max_value_in_window  max_over_p95_ratio
40   Mysql01    Mysql-MySQL_3306_Innodb buffer pool pages dirty            30    14.00000                30                    1 2021-03-25 20:30:00+08:00              85.0000            5.071429
82   Mysql01               Mysql-MySQL_3306_Max trx rows locked            30     0.55000                30                    2 2021-03-25 20:07:00+08:00               1.0000            0.818182
84   Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified            30     0.55000                30                    2 2021-03-25 20:07:00+08:00               1.0000            0.818182
123  Mysql01         Mysql-MySQL_3306_max trx lock memory bytes            30   624.80000                30                    2 2021-03-25 20:07:00+08:00            1136.0000            0.818182
306  Mysql02               Mysql-MySQL_3306_Max trx rows locked            30     0.55000                30                    2 2021-03-25 20:12:00+08:00               1.0000            0.818182
180  Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...            10     0.22000                10                    1 2021-03-25 20:06:00+08:00               0.4000            0.818182
408  Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps            10     9.25000                10                    1 2021-03-25 20:11:00+08:00              16.0000            0.729730
199  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRTps             9     0.04002                 9                    1 2021-03-25 20:15:00+08:00               0.0667            0.666667
189  Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...             9     0.24000                 9                    1 2021-03-25 20:00:00+08:00               0.4000            0.666667
200  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRead             9     0.31998                 9                    1 2021-03-25 20:15:00+08:00               0.5333            0.666667
412  Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps             9     3.60000                 9                    1 2021-03-25 20:18:00+08:00               6.0000            0.666667
413  Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...             9     0.96000                 9                    1 2021-03-25 20:12:00+08:00               1.6000            0.666667
414  Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps             9     0.01002                 9                    1 2021-03-25 20:03:00+08:00               0.0167            0.666667
415  Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead             9     0.07998                 9                    1 2021-03-25 20:03:00+08:00               0.1333            0.666667
346  Mysql02                    Mysql-MySQL_3306_ThreadsRunning            30     6.10000                30                    2 2021-03-25 20:04:00+08:00              10.0000            0.639344
403  Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             8     1.95000                 8                    1 2021-03-25 20:18:00+08:00               3.0000            0.538462
179  Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             9     3.40000                 9                    1 2021-03-25 20:10:00+08:00               5.0000            0.470588
126  Mysql01                    OSLinux-CPU_CPU-1_SingleCpuUtil            11     3.63890                11                    1 2021-03-25 20:18:00+08:00               5.2177            0.433867
182  Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead             9     1.49330                 9                    1 2021-03-25 20:00:00+08:00               2.1333            0.428581
133  Mysql01                            OSLinux-CPU_CPU_CPULoad            19     0.13600                19                    1 2021-03-25 20:26:00+08:00               0.1900            0.397059```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv', low_memory=False)

# Ensure timestamp_dt exists and is timezone-aware (Asia/Shanghai)
if 'timestamp_dt' not in metric_df.columns:
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    def ensure_ts_shanghai(x):
        if pd.isna(x):
            return pd.NaT
        if isinstance(x, pd.Timestamp):
            if x.tzinfo is None:
                return x.tz_localize(tz)
            else:
                return x.tz_convert(tz)
        return pd.to_datetime(x, unit='s', utc=True).tz_convert(tz)
    metric_df['timestamp_dt'] = metric_df['timestamp_dt'].apply(ensure_ts_shanghai)

# Candidate components
candidates = ['Mysql01','Mysql02','MG01','MG02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'IG01','IG02','Redis01','Redis02','apache01','apache02']

# Restrict to candidate components
mdf = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# 1) Global stats per (cmdb_id, kpi_name) using full series
global_stats = (
    mdf
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(total_points='count', global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter to incident window
start = pd.Timestamp('2021-03-25 20:00:00', tz=tz)
end = pd.Timestamp('2021-03-25 20:30:00', tz=tz)
window = mdf[(mdf['timestamp_dt'] >= start) & (mdf['timestamp_dt'] <= end)].copy()

# Merge global thresholds into window rows
w = window.merge(global_stats, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

# mark exceedances (strictly greater than global_p95)
w['exceed_flag'] = w['value'] > w['global_p95']

# Per-(component,kpi) window stats
points = w.groupby(['cmdb_id','kpi_name'], dropna=False).size().rename('points_in_window').reset_index()
max_vals = w.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].max().rename('max_value_in_window').reset_index()

exceed_rows = w[w['exceed_flag']].copy()
if not exceed_rows.empty:
    exceed_stats = exceed_rows.groupby(['cmdb_id','kpi_name'], dropna=False).agg(
        exceed_count_window=('exceed_flag','sum'),
        first_exceed_timestamp=('timestamp_dt','min')
    ).reset_index()
else:
    exceed_stats = pd.DataFrame(columns=['cmdb_id','kpi_name','exceed_count_window','first_exceed_timestamp'])

# Combine per-kpi stats with global stats to ensure full coverage
group_all = global_stats.merge(points, on=['cmdb_id','kpi_name'], how='left') \
                        .merge(max_vals, on=['cmdb_id','kpi_name'], how='left') \
                        .merge(exceed_stats, on=['cmdb_id','kpi_name'], how='left')

group_all['points_in_window'] = group_all['points_in_window'].fillna(0).astype(int)
group_all['exceed_count_window'] = group_all['exceed_count_window'].fillna(0).astype(int)

# Ensure first_exceed_timestamp tz-aware (it comes from timestamp_dt so should already be tz-aware or NaT)
def _fix_ts(ts):
    if pd.isna(ts):
        return pd.NaT
    if isinstance(ts, pd.Timestamp):
        if ts.tzinfo is None:
            return ts.tz_localize(tz)
        else:
            return ts.tz_convert(tz)
    return pd.to_datetime(ts).tz_localize(tz)

group_all['first_exceed_timestamp'] = group_all['first_exceed_timestamp'].apply(_fix_ts)

# 3) Compute per-(component,kpi) ratio when exceed_count_window > 0 and global_p95 != 0
def compute_ratio(row):
    if row['exceed_count_window'] <= 0:
        return np.nan
    gp95 = row['global_p95']
    maxv = row['max_value_in_window']
    if pd.isna(gp95) or gp95 == 0 or pd.isna(maxv):
        return np.nan
    return (maxv - gp95) / gp95

group_all['kpi_over_p95_ratio'] = group_all.apply(compute_ratio, axis=1)

# 4) Aggregate per-component
# earliest_exceed_timestamp per component
comp_earliest = (
    group_all[group_all['exceed_count_window'] > 0]
    .groupby('cmdb_id', dropna=False)
    .agg(earliest_exceed_timestamp=('first_exceed_timestamp','min'))
    .reset_index()
)

# distinct_kpis_with_exceed and total_exceed_points
comp_counts = (
    group_all[group_all['exceed_count_window'] > 0]
    .groupby('cmdb_id', dropna=False)
    .agg(distinct_kpis_with_exceed=('kpi_name','nunique'),
         total_exceed_points=('exceed_count_window','sum'))
    .reset_index()
)

# max_over_p95_ratio and top_kpi_by_ratio (ignore NaN ratios)
ratios = group_all[['cmdb_id','kpi_name','kpi_over_p95_ratio']].copy()
ratios_nonan = ratios[~ratios['kpi_over_p95_ratio'].isna()].copy()
if not ratios_nonan.empty:
    idx = ratios_nonan.groupby('cmdb_id')['kpi_over_p95_ratio'].idxmax()
    comp_ratio = ratios_nonan.loc[idx].rename(columns={'kpi_over_p95_ratio':'max_over_p95_ratio','kpi_name':'top_kpi_by_ratio'})[['cmdb_id','top_kpi_by_ratio','max_over_p95_ratio']].reset_index(drop=True)
else:
    comp_ratio = pd.DataFrame(columns=['cmdb_id','top_kpi_by_ratio','max_over_p95_ratio'])

# Combine into final per-component summary, include all candidates
comp_df = pd.DataFrame({'cmdb_id': candidates})
comp_df = comp_df.merge(comp_earliest, on='cmdb_id', how='left')
comp_df = comp_df.merge(comp_counts, on='cmdb_id', how='left')
comp_df = comp_df.merge(comp_ratio, on='cmdb_id', how='left')

comp_df['distinct_kpis_with_exceed'] = comp_df['distinct_kpis_with_exceed'].fillna(0).astype(int)
comp_df['total_exceed_points'] = comp_df['total_exceed_points'].fillna(0).astype(int)
comp_df['top_kpi_by_ratio'] = comp_df['top_kpi_by_ratio'].where(comp_df['top_kpi_by_ratio'].notna(), None)

# Sort by earliest_exceed_timestamp ascending (NaT last), tie-breaker by max_over_p95_ratio descending
comp_df_for_sort = comp_df.copy()
comp_df_for_sort['ratio_for_sort'] = comp_df_for_sort['max_over_p95_ratio'].fillna(-np.inf)
comp_sorted = comp_df_for_sort.sort_values(by=['earliest_exceed_timestamp','ratio_for_sort'], ascending=[True, False], na_position='last')

# Select required columns and limit to 20 rows
comp_timeline_summary = comp_sorted[['cmdb_id','earliest_exceed_timestamp','distinct_kpis_with_exceed','total_exceed_points','max_over_p95_ratio','top_kpi_by_ratio']].head(20)

comp_timeline_summary

```
Out[7]:
```
```python
# Plain-English compact summary of the per-component anomaly timeline
# Reuses comp_timeline_summary computed earlier in the session.

try:
    comp_timeline_summary
except NameError:
    raise RuntimeError("comp_timeline_summary not found. Please run the anomaly timeline aggregation step first.")

# Build compact summary
df = comp_timeline_summary.copy()

# Prepare readable timestamps
df['earliest_str'] = df['earliest_exceed_timestamp'].astype(str).replace('NaT','None')

# Top components by max_over_p95_ratio
top_by_ratio = df.dropna(subset=['max_over_p95_ratio']).sort_values('max_over_p95_ratio', ascending=False).head(5)

# Top components by total_exceed_points
top_by_exceeds = df.sort_values('total_exceed_points', ascending=False).head(5)

lines = []
lines.append("Anomaly timeline summary (2021-03-25 20:00–20:30 Asia/Shanghai):")
lines.append("")
# Earliest exceed overview
earliest_nonnull = df[df['earliest_exceed_timestamp'].notna()].sort_values('earliest_exceed_timestamp').head(6)
lines.append("Earliest exceed events (components with earliest exceed timestamps):")
for _, r in earliest_nonnull.iterrows():
    lines.append(f"- {r['cmdb_id']}: earliest_exceed={r['earliest_str']}, distinct_kpis_with_exceed={r['distinct_kpis_with_exceed']}, total_exceed_points={r['total_exceed_points']}")
lines.append("")

# Top by ratio
lines.append("Top components by relative spike above global-P95 (max_over_p95_ratio):")
for _, r in top_by_ratio.iterrows():
    ratio = r['max_over_p95_ratio']
    lines.append(f"- {r['cmdb_id']}: ratio={ratio:.3f}, top_kpi={r['top_kpi_by_ratio']}, earliest_exceed={r['earliest_str']}")

lines.append("")

# Top by total exceed points
lines.append("Top components by total exceed points in window:")
for _, r in top_by_exceeds.iterrows():
    lines.append(f"- {r['cmdb_id']}: total_exceed_points={r['total_exceed_points']}, distinct_kpis_with_exceed={r['distinct_kpis_with_exceed']}, earliest_exceed={r['earliest_str']}")

lines.append("")
lines.append("Key takeaways:")
lines.append("- Many core components showed exceedances right at or shortly after 20:00 (most earliest_exceed timestamps are 20:00–20:01).")
lines.append("- Biggest relative spike (by ratio) is Redis02 (driven by CPU WIO), followed by Mysql01 (driven by 'Innodb buffer pool pages dirty').")
lines.append("- Mysql02 and Mysql01 have the highest total_exceed_points (Mysql02=152, Mysql01=135), indicating broad KPI-level anomalies on DB nodes.")
lines.append("- Common signal types driving ratios: CPU WIO, disk I/O (DSKRead/DSKBps/DSKTps), and MySQL-specific metrics (locks, dirty pages).")
lines.append("- If global_p95 == 0 for a KPI, that KPI's ratio was treated as undefined and excluded from the component max (i.e., ignored).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id earliest_exceed_timestamp  distinct_kpis_with_exceed  total_exceed_points  max_over_p95_ratio                                   top_kpi_by_ratio
11   Redis02 2021-03-25 20:00:00+08:00                         33                   38            9.420361                             OSLinux-CPU_CPU_CPUWio
0    Mysql01 2021-03-25 20:00:00+08:00                         94                  135            5.071429    Mysql-MySQL_3306_Innodb buffer pool pages dirty
5   Tomcat02 2021-03-25 20:00:00+08:00                         42                   50            3.414892                             OSLinux-CPU_CPU_CPUWio
4   Tomcat01 2021-03-25 20:00:00+08:00                         46                   54            2.614010    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead
2       MG01 2021-03-25 20:00:00+08:00                         42                   50            2.009119    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead
3       MG02 2021-03-25 20:00:00+08:00                         47                   56            1.515780                             OSLinux-CPU_CPU_CPUWio
6   Tomcat03 2021-03-25 20:00:00+08:00                         43                   52            1.469136    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead
12  apache01 2021-03-25 20:00:00+08:00                         30                   30            1.292166                             OSLinux-CPU_CPU_CPUWio
8       IG01 2021-03-25 20:00:00+08:00                         38                   42            0.927711     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps
9       IG02 2021-03-25 20:00:00+08:00                         34                   36            0.818182     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps
13  apache02 2021-03-25 20:00:00+08:00                         29                   29            0.818182     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps
7   Tomcat04 2021-03-25 20:01:00+08:00                         40                   48            2.333333  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...
1    Mysql02 2021-03-25 20:01:00+08:00                        103                  152            0.818182               Mysql-MySQL_3306_Max trx rows locked
10   Redis01 2021-03-25 20:04:00+08:00                         28                   28            0.559499                             OSLinux-CPU_CPU_CPUWio```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# timezone
tz = pytz.timezone('Asia/Shanghai')

# reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv', low_memory=False)

# ensure timestamp_dt exists and is timezone-aware (Asia/Shanghai)
if 'timestamp_dt' not in metric_df.columns:
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    def _ensure_shanghai(ts):
        if pd.isna(ts):
            return pd.NaT
        if isinstance(ts, pd.Timestamp):
            if ts.tzinfo is None:
                return ts.tz_localize(tz)
            else:
                return ts.tz_convert(tz)
        return pd.to_datetime(ts, unit='s', utc=True).tz_convert(tz)
    metric_df['timestamp_dt'] = metric_df['timestamp_dt'].apply(_ensure_shanghai)

# candidate components
candidates = ['Mysql01','Mysql02','Redis02','Tomcat04','MG01','MG02','IG01','IG02']

# 1) compute global P95 per (cmdb_id, kpi_name) using full series (global thresholds before filtering)
global_stats = (
    metric_df[metric_df['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(total_points='count', global_p95=lambda s: s.quantile(0.95))
    .reset_index()
)

# 2) filter to incident window (inclusive)
start = pd.Timestamp('2021-03-25 20:00:00', tz=tz)
end = pd.Timestamp('2021-03-25 20:30:00', tz=tz)
window = metric_df[
    (metric_df['cmdb_id'].isin(candidates)) &
    (metric_df['timestamp_dt'] >= start) &
    (metric_df['timestamp_dt'] <= end)
].copy()

# merge global thresholds into window rows
w = window.merge(global_stats, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

# mark exceedances (strict > global_p95)
w['exceed_flag'] = w['value'] > w['global_p95']

# per-(component,kpi) window stats
per_kpi = (
    w.groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(
        points_in_window=('value','size'),
        exceed_count_window=('exceed_flag','sum'),
        first_exceed_timestamp=('timestamp_dt', lambda x: x[w.loc[x.index,'exceed_flag']].min() if (w.loc[x.index,'exceed_flag'].any()) else pd.NaT),
        max_value_in_window=('value','max'),
        global_p95=('global_p95','first')
    )
    .reset_index()
)

# ensure numeric types
per_kpi['points_in_window'] = per_kpi['points_in_window'].fillna(0).astype(int)
per_kpi['exceed_count_window'] = per_kpi['exceed_count_window'].fillna(0).astype(int)
per_kpi['first_exceed_timestamp'] = pd.to_datetime(per_kpi['first_exceed_timestamp']).dt.tz_convert(tz)

# 3) compute kpi_over_p95_ratio where exceed_count_window > 0 and global_p95 != 0
def kpi_ratio(row):
    if row['exceed_count_window'] <= 0:
        return np.nan
    gp95 = row['global_p95']
    maxv = row['max_value_in_window']
    if pd.isna(gp95) or gp95 == 0 or pd.isna(maxv):
        return np.nan
    return (maxv - gp95) / gp95

per_kpi['kpi_over_p95_ratio'] = per_kpi.apply(kpi_ratio, axis=1)

# 4) aggregate per-component
# earliest_exceed_timestamp
comp_earliest = (
    per_kpi[per_kpi['exceed_count_window'] > 0]
    .groupby('cmdb_id', dropna=False)['first_exceed_timestamp']
    .min()
    .reset_index()
    .rename(columns={'first_exceed_timestamp':'earliest_exceed_timestamp'})
)

# distinct_kpis_with_exceed and total_exceed_points
comp_counts = (
    per_kpi[per_kpi['exceed_count_window'] > 0]
    .groupby('cmdb_id', dropna=False)
    .agg(distinct_kpis_with_exceed=('kpi_name','nunique'),
         total_exceed_points=('exceed_count_window','sum'))
    .reset_index()
)

# component_max_over_p95_ratio: max of kpi_over_p95_ratio per component (ignore NaN)
comp_ratio = (
    per_kpi.dropna(subset=['kpi_over_p95_ratio'])
    .groupby('cmdb_id', dropna=False)['kpi_over_p95_ratio']
    .max()
    .reset_index()
    .rename(columns={'kpi_over_p95_ratio':'component_max_over_p95_ratio'})
)

# combine into final summary for all candidates (include components with no exceeds)
comp_df = pd.DataFrame({'cmdb_id': candidates})
comp_df = comp_df.merge(comp_earliest, on='cmdb_id', how='left')
comp_df = comp_df.merge(comp_counts, on='cmdb_id', how='left')
comp_df = comp_df.merge(comp_ratio, on='cmdb_id', how='left')

# fill zeros where appropriate
comp_df['distinct_kpis_with_exceed'] = comp_df['distinct_kpis_with_exceed'].fillna(0).astype(int)
comp_df['total_exceed_points'] = comp_df['total_exceed_points'].fillna(0).astype(int)
# component_max_over_p95_ratio may be NaN if no valid ratios; keep NaN

# 5) severity_score = total_exceed_points * component_max_over_p95_ratio (treat NaN ratio as 0 for score)
comp_df['severity_score'] = comp_df['total_exceed_points'] * comp_df['component_max_over_p95_ratio'].fillna(0)

# ordering: earliest_exceed_timestamp asc (NaT last), tie-breaker severity_score desc
comp_df_for_sort = comp_df.copy()
comp_df_for_sort['severity_for_sort'] = comp_df_for_sort['severity_score']
comp_sorted = comp_df_for_sort.sort_values(by=['earliest_exceed_timestamp','severity_for_sort'], ascending=[True, False], na_position='last')

# select required columns and keep compact
severity_df = comp_sorted[['cmdb_id','earliest_exceed_timestamp','distinct_kpis_with_exceed','total_exceed_points','component_max_over_p95_ratio','severity_score']].head(20)

severity_df

```
Out[9]:
```
```python
# Plain-English summary string of the computed severity ranking (compact)
summary = (
    "Severity ranking summary (metric global-P95, window 2021-03-25 20:00–20:30 Asia/Shanghai):\n\n"
    "Rank order (by earliest exceed timestamp, tie-broken by severity score):\n"
    "1) Mysql01 — earliest_exceed=2021-03-25 20:00, distinct_kpis_with_exceed=94, "
    "total_exceed_points=135, component_max_over_p95_ratio≈5.071, severity_score≈684.64 (highest severity)\n"
    "2) Redis02 — earliest_exceed=2021-03-25 20:00, distinct_kpis_with_exceed=33, "
    "total_exceed_points=38, component_max_over_p95_ratio≈9.420, severity_score≈357.97\n"
    "3) MG01    — earliest_exceed=2021-03-25 20:00, distinct_kpis_with_exceed=42, "
    "total_exceed_points=50, component_max_over_p95_ratio≈2.009, severity_score≈100.46\n"
    "4) MG02    — earliest_exceed=2021-03-25 20:00, distinct_kpis_with_exceed=47, "
    "total_exceed_points=56, component_max_over_p95_ratio≈1.516, severity_score≈84.88\n"
    "5) IG01    — earliest_exceed=2021-03-25 20:00, distinct_kpis_with_exceed=38, "
    "total_exceed_points=42, component_max_over_p95_ratio≈0.928, severity_score≈38.96\n"
    "6) IG02    — earliest_exceed=2021-03-25 20:00, distinct_kpis_with_exceed=34, "
    "total_exceed_points=36, component_max_over_p95_ratio≈0.818, severity_score≈29.45\n"
    "7) Mysql02 — earliest_exceed=2021-03-25 20:01, distinct_kpis_with_exceed=103, "
    "total_exceed_points=152, component_max_over_p95_ratio≈0.818, severity_score≈124.36\n"
    "8) Tomcat04— earliest_exceed=2021-03-25 20:01, distinct_kpis_with_exceed=40, "
    "total_exceed_points=48, component_max_over_p95_ratio≈2.333, severity_score≈112.00\n\n"
    "Key takeaways:\n"
    "- Mysql01 is the top severity signal driven by a very large relative spike (max_over_p95_ratio ~5.07) across many KPIs.\n"
    "- Redis02 shows the largest relative spike (ratio ~9.42) but with fewer total exceed points, placing it second by severity score.\n"
    "- MG01 and MG02 show substantial multi-KPI exceedances with moderate ratios, contributing to mid-level severity.\n"
    "- Mysql02 and Tomcat04 have later earliest exceeds (20:01) and significant total exceed counts, but lower max ratios, so their severity scores are lower than Mysql01/Redis02.\n"
    "- Metric ratios where a KPI's global_p95 == 0 were excluded from ratio-based max calculations (treated as undefined) and not used in the component max ratio.\n\n"
    "Recommendation: prioritize investigation of Mysql01 and Redis02 first (highest severity), then MG01/MG02, followed by Tomcat04 and the IG nodes."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id earliest_exceed_timestamp  distinct_kpis_with_exceed  total_exceed_points  component_max_over_p95_ratio  severity_score
0   Mysql01 2021-03-25 20:00:00+08:00                         94                  135                      5.071429      684.642857
2   Redis02 2021-03-25 20:00:00+08:00                         33                   38                      9.420361      357.973733
4      MG01 2021-03-25 20:00:00+08:00                         42                   50                      2.009119      100.455927
5      MG02 2021-03-25 20:00:00+08:00                         47                   56                      1.515780       84.883666
6      IG01 2021-03-25 20:00:00+08:00                         38                   42                      0.927711       38.963855
7      IG02 2021-03-25 20:00:00+08:00                         34                   36                      0.818182       29.454545
1   Mysql02 2021-03-25 20:01:00+08:00                        103                  152                      0.818182      124.363636
3  Tomcat04 2021-03-25 20:01:00+08:00                         40                   48                      2.333333      112.000000```
```